# Ancestry Mapper Version 1

**Important:** `ancestry_worker.py` must be in the same folder as this notebook.

Input file requirements (see `input.txt` for an example):
* Higher classification strings must start & end with pipes
* The taxon name should NOT be part of its higher classification string (except in rare cases where the parent taxon has the same name as the taxon)

Notes:

Some higher classification strings will end up with multiple Index matches. These are resolved automatically during the mapping process by choosing the Index value corresponding to the rightmost match in the higher classification string. This will be appropriate most of the time, but we need to keep an eye on these multiple matches to make sure they are compatible. If we get incompatible matches, Katja needs to modify the Ancestry Index mappings to prevent such matches in the future.
* The `compatibleAncestors.txt` file documents pairs of matches that are compatible and can be ignored. The sequence of values in the pair does not matter, i.e., both Arthropoda; Insecta and Insecta; Arthropoda are compatible although only Arthropoda; Insecta is listed in the file.
* The `output_multimatches.tsv` file documents the lines with multiple matches before they have been fixed. We need to check the pairs of matches in this file against the `compatibleAncestors.txt` file. If there are any pairs that are not on the compatible list, please send those line from the `output_multimatches.tsv` file as `incompatible_multimatches.tsv` to Katja.

## Step 1 — Import tools

In [1]:
import re
import csv
import time
import multiprocessing

# Import worker functions from the separate file.
from ancestry_worker import _init_worker, process_chunk

## Step 2 — Set your file paths

In [4]:
TAXONOMY_FILE    = "input.txt"              # the higherClassification strings we want to map to the index
MAPPING_FILE     = "ancestryIndex.tsv"        # the mapping file with the regexes from the AncestryIndex doc
OUTPUT_FILE      = "mappedAncestryStrings.tsv"             # primary output: one best label per line
MULTIMATCH_FILE  = "output_multimatches.tsv" # lines that matched 2+ rules (all labels shown)

NUM_WORKERS = 6                             # 6 of my 8 cores; leaves two free for the OS

## Step 3 — Load the mapping rules

In [6]:
raw_rules = []

with open(MAPPING_FILE, "r", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        if len(row) < 2:
            continue
        label   = row[0].strip()
        pattern = row[1].strip()
        try:
            re.compile(pattern)
            raw_rules.append((label, pattern))
        except re.error as e:
            print(f"Warning: skipping bad regex for label '{label}': {e}")

print(f"Loaded {len(raw_rules)} mapping rules.")

Loaded 5131 mapping rules.


## Step 4 — Process the taxonomy file in parallel

In [8]:
start = time.time()

# Read all lines at once
print("Reading taxonomy file...")
with open(TAXONOMY_FILE, "r", encoding="utf-8") as f:
    all_lines = f.readlines()
print(f"  {len(all_lines):,} lines loaded.")

# Split into chunks, one per worker
chunk_size = len(all_lines) // NUM_WORKERS + 1
chunks = [all_lines[i : i + chunk_size] for i in range(0, len(all_lines), chunk_size)]
print(f"  Split into {len(chunks)} chunks of ~{chunk_size:,} lines each.")

# Launch the worker pool
# process_chunk now returns (primary_results, multimatch_results) per chunk
print(f"\nProcessing with {NUM_WORKERS} parallel workers...")
with multiprocessing.Pool(
    processes=NUM_WORKERS,
    initializer=_init_worker,
    initargs=(raw_rules,)
) as pool:
    result_chunks = pool.map(process_chunk, chunks)

# Write primary output (one best label per line)
print("Writing primary output...")
lines_matched = 0
with open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:
    outfile.write("taxonomy_string\tlabel\n")
    for primary, _ in result_chunks:
        for line in primary:
            outfile.write(line)
            if line.split("\t")[1].strip():
                lines_matched += 1

# Write multi-match output (all labels, semicolon-separated, only for 2+ matches)
print("Writing multi-match output...")
lines_multimatch = 0
with open(MULTIMATCH_FILE, "w", encoding="utf-8") as outfile:
    outfile.write("taxonomy_string\tlabels\n")
    for _, multimatch in result_chunks:
        for line in multimatch:
            outfile.write(line)
            lines_multimatch += 1

elapsed = time.time() - start
print(f"\nDone in {elapsed/60:.1f} minutes.")
print(f"Primary results written to:    {OUTPUT_FILE}")
print(f"Multi-match results written to: {MULTIMATCH_FILE}")
print(f"Lines with at least one match:  {lines_matched:,}")
print(f"Lines with multiple matches:    {lines_multimatch:,}")

Reading taxonomy file...
  39 lines loaded.
  Split into 6 chunks of ~7 lines each.

Processing with 6 parallel workers...
Writing primary output...
Writing multi-match output...

Done in 0.0 minutes.
Primary results written to:    mappedAncestryStrings.tsv
Multi-match results written to: output_multimatches.tsv
Lines with at least one match:  39
Lines with multiple matches:    28


## Step 5 — Preview the primary output

In [11]:
print(f"=== {OUTPUT_FILE} (first 11 lines) ===")
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line, end="")
        if i >= 10:
            break

=== mappedAncestryStrings.tsv (first 11 lines) ===
taxonomy_string	label
|Life|Cellular Organisms|Eukaryota|Opisthokonta|Metazoa|Bilateria|Protostomia|Ecdysozoa|Arthropoda|Pancrustacea|Hexapoda|Insecta|Pterygota|Neoptera|Polyneoptera|Plecoptera|Perloidea|Perlidae|Miniperla|	Plecoptera
|Life|Cellular Organisms|Eukaryota|Opisthokonta|Metazoa|Bilateria|Protostomia|Ecdysozoa|Arthropoda|Pancrustacea|Hexapoda|Insecta|Pterygota|Neoptera|Endopterygota|Diptera|Axymyiidae|Protaxymyia|	Diptera
|Life|Cellular Organisms|Eukaryota|Opisthokonta|Metazoa|Bilateria|Protostomia|Ecdysozoa|Arthropoda|Pancrustacea|Hexapoda|Insecta|Pterygota|Neoptera|Endopterygota|Coleoptera|Adephaga|Carabidae|Abacetus|	Coleoptera
|Life|Cellular Organisms|Eukaryota|Opisthokonta|Metazoa|Bilateria|Protostomia|Ecdysozoa|Arthropoda|Pancrustacea|Hexapoda|Insecta|Pterygota|Neoptera|Polyneoptera|Orthopterida|Orthoptera|Caelifera|Acridoidea|Acrididae|Acanthoxia|	Orthoptera
|Life|Cellular Organisms|Eukaryota|Opisthokonta|Metazoa|Bila

## Step 6 — Preview the multi-match output

In [13]:
print(f"=== {MULTIMATCH_FILE} (first 11 lines) ===")
with open(MULTIMATCH_FILE, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line, end="")
        if i >= 10:
            break

=== output_multimatches.tsv (first 11 lines) ===
taxonomy_string	labels
|Life|Cellular Organisms|Eukaryota|Opisthokonta|Metazoa|Bilateria|Protostomia|Ecdysozoa|Arthropoda|Pancrustacea|Hexapoda|Insecta|Pterygota|Neoptera|Polyneoptera|Plecoptera|Perloidea|Perlidae|Miniperla|	Insecta; Plecoptera
|Life|Cellular Organisms|Eukaryota|Opisthokonta|Metazoa|Bilateria|Protostomia|Ecdysozoa|Arthropoda|Pancrustacea|Hexapoda|Insecta|Pterygota|Neoptera|Endopterygota|Diptera|Axymyiidae|Protaxymyia|	Diptera; Insecta
|Life|Cellular Organisms|Eukaryota|Opisthokonta|Metazoa|Bilateria|Protostomia|Ecdysozoa|Arthropoda|Pancrustacea|Hexapoda|Insecta|Pterygota|Neoptera|Endopterygota|Coleoptera|Adephaga|Carabidae|Abacetus|	Coleoptera; Insecta
|Life|Cellular Organisms|Eukaryota|Opisthokonta|Metazoa|Bilateria|Protostomia|Ecdysozoa|Arthropoda|Pancrustacea|Hexapoda|Insecta|Pterygota|Neoptera|Polyneoptera|Orthopterida|Orthoptera|Caelifera|Acridoidea|Acrididae|Acanthoxia|	Insecta; Orthoptera
|Life|Cellular Organisms|

## Check results

### Unmatched lines

In [15]:
! awk -F '\t' '{if($2 == ""){print}}' mappedAncestryStrings.tsv | wc -l

       0


In [17]:
! awk -F '\t' '{if($2 == ""){print}}' mappedAncestryStrings.tsv

### Check multiple match groups

In [19]:
! awk -F'\t' '$2 ~ /;/ {count[$2]++} END {for (s in count) print s "\t" count[s]}' output_multimatches.tsv

Diptera; Insecta	6
Angiosperms; Fungi	1
Insecta; Plecoptera	1
Coleoptera; Insecta	5
Insecta; Orthoptera	15
